# 🏥 Final Model Retraining v4 - Data Sanitization + Monotonic Constraints

**Problem:** Previous models learned from 273,688 toxic rows where insulin was given during dangerous scenarios (Hypo, Rapid Drop, etc.)

**Solution:**
1. **Data Sanitization:** Force `bolus = 0.0` for all toxic scenarios (BOTH Train + Val)
2. **Monotonic Constraints:** Enforce physics-based relationships (CGM↑ → Dose↑)
3. **Retrain:** Create `xgboost_final_model_v4_corrected.json`

## 1. Setup & Dependencies

In [ ]:
# Install dependencies (Colab)
!pip install polars xgboost -q

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import polars as pl
import xgboost as xgb
import numpy as np
import os

print(f"Polars: {pl.__version__}")
print(f"XGBoost: {xgb.__version__}")

In [ ]:
# Configuration
BASE_PATH = "/content/drive/MyDrive/DiabetesKDS/data/processed"

FILES = {
    "train": f"{BASE_PATH}/train_augmented_fixed.parquet",
    "val": f"{BASE_PATH}/test_ready_for_xgboost.parquet",
    "output_model": f"{BASE_PATH}/xgboost_final_model_v4_corrected.json",
    "output_data": f"{BASE_PATH}/train_sanitized_v4.parquet"
}

# Verify files exist
for name, path in FILES.items():
    if "output" not in name:
        exists = "✅" if os.path.exists(path) else "❌"
        print(f"{exists} {name}: {path}")

## 2. Define Toxic Mask (Reusable Function)

In [ ]:
def get_toxic_mask():
    """Define toxic scenarios mask - SAME logic for Train & Val."""
    return (
        # 1. Falling & Low: CGM < 110, Delta < -5, No Carbs
        ((pl.col("CGM") < 110) & 
         (pl.col("cgm_delta_15min") < -5) & 
         (pl.col("carbs") == 0)) |
        
        # 2. Hypo Hazard: CGM < 70, No Carbs (MOST DANGEROUS)
        ((pl.col("CGM") < 70) & 
         (pl.col("carbs") == 0)) |
        
        # 3. Rapid Drop: Delta < -15, No Carbs
        ((pl.col("cgm_delta_15min") < -15) & 
         (pl.col("carbs") == 0)) |
        
        # 4. Stable Low: 70 ≤ CGM < 100, Stable, No Carbs
        ((pl.col("CGM") >= 70) & 
         (pl.col("CGM") < 100) & 
         (pl.col("cgm_delta_15min").abs() < 3) & 
         (pl.col("carbs") == 0))
    )

def sanitize_dataframe(df, name="Dataset"):
    """Apply toxic cleaning to a dataframe."""
    toxic_mask = get_toxic_mask()
    n_toxic = df.filter(toxic_mask & (pl.col("bolus") > 0)).shape[0]
    
    df_clean = df.with_columns(
        pl.when(toxic_mask)
        .then(0.0)
        .otherwise(pl.col("bolus"))
        .alias("bolus")
    )
    
    print(f"   {name}: Corrected {n_toxic:,} toxic rows")
    return df_clean, n_toxic

print("✅ Toxic mask function defined")

## 3. Load & Sanitize Training Data

In [ ]:
# Load training data
print("📂 Loading training data...")
df_train_raw = pl.read_parquet(FILES["train"])
print(f"✅ Loaded: {df_train_raw.shape[0]:,} rows × {df_train_raw.shape[1]} columns")

# Sanitize training data
print("\n🧹 SANITIZING TRAINING DATA...")
df_clean, n_train_toxic = sanitize_dataframe(df_train_raw, "Training")

# Verify
post_fix = df_clean.filter(get_toxic_mask() & (pl.col("bolus") > 0)).shape[0]
print(f"✅ Verification: Toxic rows with bolus > 0 = {post_fix} (should be 0)")

In [ ]:
# Save sanitized training data
print(f"\n💾 Saving sanitized data to: {FILES['output_data']}")
df_clean.write_parquet(FILES['output_data'])
print("✅ Saved!")

## 4. Load & Sanitize Validation Data

In [ ]:
# Load validation data
print("📂 Loading validation data...")
df_val_raw = pl.read_parquet(FILES["val"])
print(f"✅ Loaded: {df_val_raw.shape[0]:,} rows")

# ⭐ CRITICAL: Apply SAME toxic cleaning to validation set
print("\n🧹 SANITIZING VALIDATION DATA (Same logic as Training)...")
df_val, n_val_toxic = sanitize_dataframe(df_val_raw, "Validation")

# Verify
post_fix_val = df_val.filter(get_toxic_mask() & (pl.col("bolus") > 0)).shape[0]
print(f"✅ Verification: Toxic rows with bolus > 0 = {post_fix_val} (should be 0)")

print(f"\n📊 TOTAL TOXIC ROWS CORRECTED: {n_train_toxic + n_val_toxic:,}")

## 5. Define Features & Monotonic Constraints

In [ ]:
# Feature configuration
FEATURES = [
    'CGM',                  # 0: Current glucose
    'carbs',                # 1: Carbohydrates
    'time_sin',             # 2: Time of day (sin)
    'time_cos',             # 3: Time of day (cos)
    'is_weekend',           # 4: Weekend flag
    'cgm_delta_5min',       # 5: 5-min trend
    'cgm_delta_15min',      # 6: 15-min trend
    'basal',                # 7: Basal rate
    'age',                  # 8: Patient age
    'weight',               # 9: Patient weight
    'gender_encoded',       # 10: Gender
    'insulin_type_encoded'  # 11: Insulin type
]

TARGET = 'bolus'

# Monotonic Constraints (THE PHYSICS ENGINE)
MONOTONE_CONSTRAINTS = (
    1,   # CGM: Higher glucose → More insulin ✓
    1,   # carbs: More carbs → More insulin ✓
    0,   # time_sin: No constraint
    0,   # time_cos: No constraint
    0,   # is_weekend: No constraint
    1,   # cgm_delta_5min: Rising trend → More insulin ✓
    1,   # cgm_delta_15min: Rising trend → More insulin ✓
    1,   # basal: Higher basal → More overall insulin need ✓
    0,   # age: Non-linear patterns allowed
    0,   # weight: Non-linear patterns allowed
    0,   # gender: Categorical
    0    # insulin_type: Categorical
)

print("📋 Feature → Constraint Mapping:")
print("-" * 40)
for feat, constraint in zip(FEATURES, MONOTONE_CONSTRAINTS):
    direction = {1: "↑ Increasing", 0: "○ None", -1: "↓ Decreasing"}[constraint]
    print(f"   {feat:25} → {direction}")

## 6. Prepare DMatrix Objects

In [ ]:
# Check available features
available = [f for f in FEATURES if f in df_clean.columns]
missing = [f for f in FEATURES if f not in df_clean.columns]

print(f"✅ Available features: {len(available)}/{len(FEATURES)}")
if missing:
    print(f"⚠️ Missing features: {missing}")

# Prepare matrices
print("\n🔧 Preparing DMatrix objects...")

# Training data
X_train = df_clean.select(available).to_numpy().astype(np.float32)
y_train = df_clean[TARGET].to_numpy().astype(np.float32)

# Validation data (NOW SANITIZED!)
X_val = df_val.select(available).to_numpy().astype(np.float32)
y_val = df_val[TARGET].to_numpy().astype(np.float32)

# Create DMatrix
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=available)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=available)

print(f"✅ Train DMatrix: {dtrain.num_row():,} × {dtrain.num_col()}")
print(f"✅ Val DMatrix: {dval.num_row():,} × {dval.num_col()}")

# Class balance
n_zero = (y_train == 0).sum()
n_pos = (y_train > 0).sum()
imbalance = n_zero / n_pos if n_pos > 0 else 1
print(f"\n📊 Class Balance: {n_zero:,} zeros / {n_pos:,} positive (ratio: {imbalance:.2f})")

## 7. Training Configuration

In [ ]:
# XGBoost Parameters with Monotonic Constraints
PARAMS = {
    # Objective
    'objective': 'reg:squarederror',
    'eval_metric': 'mae',
    
    # GPU Acceleration
    'tree_method': 'hist',
    'device': 'cuda',
    
    # ⭐ MONOTONIC CONSTRAINTS (THE KEY FIX)
    'monotone_constraints': MONOTONE_CONSTRAINTS,
    
    # Class Imbalance (Lower weight - data is now cleaner)
    'scale_pos_weight': 1.5,
    'max_delta_step': 1,
    
    # Tree Parameters
    'max_depth': 8,
    'learning_rate': 0.03,
    'min_child_weight': 5,
    
    # Sampling
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    
    # Regularization
    'gamma': 0.1,
    'reg_alpha': 0.5,
    'reg_lambda': 1.0,
    
    # Other
    'seed': 42,
    'verbosity': 1
}

NUM_ROUNDS = 1000
EARLY_STOPPING = 50

print("⚙️ Training Configuration:")
print(f"   Rounds: {NUM_ROUNDS} (Early Stop: {EARLY_STOPPING})")
print(f"   Learning Rate: {PARAMS['learning_rate']}")
print(f"   Max Depth: {PARAMS['max_depth']}")
print(f"   Monotonic Constraints: ENABLED ✓")

## 8. Train Model

In [ ]:
%%time

print("🚀 STARTING TRAINING...")
print("=" * 50)

evals = [(dtrain, 'train'), (dval, 'val')]
evals_result = {}

model = xgb.train(
    params=PARAMS,
    dtrain=dtrain,
    num_boost_round=NUM_ROUNDS,
    evals=evals,
    evals_result=evals_result,
    early_stopping_rounds=EARLY_STOPPING,
    verbose_eval=100
)

print("\n" + "=" * 50)
print(f"✅ TRAINING COMPLETE!")
print(f"   Best Iteration: {model.best_iteration}")
print(f"   Best Val MAE: {model.best_score:.4f}")

In [ ]:
# Save model
print(f"\n💾 Saving model to: {FILES['output_model']}")
model.save_model(FILES['output_model'])
print("✅ Model saved!")

## 9. Sanity Check: Critical Test Cases

In [ ]:
import math

def create_test_case(cgm, carbs, cgm_delta_15min, cgm_delta_5min=None):
    """Create test feature array."""
    if cgm_delta_5min is None:
        cgm_delta_5min = cgm_delta_15min / 3
    
    hour = 12
    time_sin = math.sin(2 * math.pi * hour / 24)
    time_cos = math.cos(2 * math.pi * hour / 24)
    
    features = [
        cgm, carbs, time_sin, time_cos, 0.0,
        cgm_delta_5min, cgm_delta_15min, 0.8,
        35.0, 70.0, 0.0, 3.0
    ]
    
    return np.array([features], dtype=np.float32)


def run_test(name, cgm, carbs, delta, expected_max=0.1):
    """Run a single test case."""
    X = create_test_case(cgm, carbs, delta)
    dtest = xgb.DMatrix(X, feature_names=available)
    pred = model.predict(dtest)[0]
    
    passed = pred < expected_max
    status = "✅ PASS" if passed else "❌ FAIL"
    
    print(f"{status} | {name}")
    print(f"        CGM={cgm}, Carbs={carbs}, Delta={delta}")
    print(f"        Prediction: {pred:.4f} U (Expected < {expected_max})")
    print()
    
    return passed

In [ ]:
print("🧪 CRITICAL SAFETY TESTS")
print("=" * 50)

tests = [
    ("Hypo No Food", 65, 0, 0, 0.1),
    ("Falling Trend No Food (User's Failure Case)", 88, 0, -10, 0.1),
    ("Rapid Drop", 120, 0, -20, 0.1),
    ("Stable Low", 85, 0, 0, 0.1),
]

results = []
for name, cgm, carbs, delta, max_exp in tests:
    result = run_test(name, cgm, carbs, delta, max_exp)
    results.append(result)

print("=" * 50)
passed = sum(results)
total = len(results)
print(f"SUMMARY: {passed}/{total} tests passed")

if passed == total:
    print("\n🎉 ALL SAFETY TESTS PASSED!")
else:
    print("\n⚠️ SOME TESTS FAILED - Review model or add more safety rules")

In [ ]:
print("\n🧪 POSITIVE CASES (Should recommend insulin)")
print("=" * 50)

positive_tests = [
    ("High Glucose + Carbs", 200, 60, 5, 1.0),
    ("Very High Glucose", 280, 30, 10, 0.5),
    ("Rising Trend + Meal", 150, 45, 15, 0.3),
]

for name, cgm, carbs, delta, min_exp in positive_tests:
    X = create_test_case(cgm, carbs, delta)
    dtest = xgb.DMatrix(X, feature_names=available)
    pred = model.predict(dtest)[0]
    
    passed = pred >= min_exp
    status = "✅" if passed else "⚠️"
    
    print(f"{status} {name}: Predicted {pred:.2f} U (Expected ≥ {min_exp})")

## 10. Feature Importance

In [ ]:
import matplotlib.pyplot as plt

importance = model.get_score(importance_type='gain')
importance_df = pl.DataFrame({
    'feature': list(importance.keys()),
    'importance': list(importance.values())
}).sort('importance', descending=True)

print("📊 Feature Importance (Gain):")
print(importance_df)

plt.figure(figsize=(10, 6))
plt.barh(importance_df['feature'].to_list()[::-1], 
         importance_df['importance'].to_list()[::-1])
plt.xlabel('Importance (Gain)')
plt.title('Feature Importance - Sanitized Model v4')
plt.tight_layout()
plt.show()

## 11. Summary

In [ ]:
print("\n" + "=" * 60)
print("📋 TRAINING SUMMARY")
print("=" * 60)
print(f"""
Data Sanitization:
   - Training toxic rows corrected: {n_train_toxic:,}
   - Validation toxic rows corrected: {n_val_toxic:,}
   - Total corrected: {n_train_toxic + n_val_toxic:,}

Model Configuration:
   - Monotonic Constraints: ENABLED
   - Best Iteration: {model.best_iteration}
   - Best Val MAE: {model.best_score:.4f}

Output Files:
   - Model: {FILES['output_model']}
   - Data: {FILES['output_data']}

Safety Tests: {passed}/{total} passed
""")
print("=" * 60)